# Variabilidad Hospitalaria en la Práctica Clínica
## Proyecto Aplicado — Chile, 2019–2024

**Objetivo:** Evaluar si existe una variabilidad significativa en los protocolos de tratamiento y tiempos de estada para pacientes con diagnósticos idénticos, dependiendo del recinto hospitalario público donde son atendidos.

---
# 1. Inicio del Proyecto

## 1.1 Contexto y Problemática

La estandarización de los tratamientos médicos es fundamental para asegurar la equidad en el sistema de salud. Sin embargo, existe una problemática evidente cuando pacientes con diagnósticos similares reciben procedimientos diferentes dependiendo exclusivamente del hospital donde son atendidos. Esta investigación busca evaluar las razones subyacentes por las cuales cuadros clínicos idénticos terminan siendo gestionados de manera distinta en la red pública.

El núcleo de este estudio radica en analizar la variabilidad hospitalaria en la práctica clínica. Más allá de las brechas de acceso, el desafío analítico consiste en determinar si existe una varianza significativa en la duración de la hospitalización entre distintos recintos, después de controlar estadísticamente por la severidad clínica de los pacientes.

Comprender si algunos hospitales presentan patrones sistemáticos de mayor intervención clínica es crucial. Estas discrepancias operativas y de criterio médico generan ineficiencias estructurales que saturan la red y limitan la disponibilidad de camas y recursos. En última instancia, esta variabilidad crea una brecha de calidad que afecta desproporcionadamente a los sectores con menos recursos, quienes dependen de la eficiencia del sistema público. Utilizar el análisis de datos para visibilizar estas diferencias es un paso esencial para optimizar la gestión de salud y asegurar un estándar de atención equitativo, independientemente de dónde se trate el paciente.

## 1.2 Pregunta de Investigación

> **¿Existe una variabilidad significativa en los protocolos de tratamiento y tiempos de estada para pacientes con diagnósticos idénticos, dependiendo de si son atendidos en uno u otro recinto de la red hospitalaria pública?**

## 1.3 Identificación de Datos

Para abordar empíricamente esta problemática, la investigación se fundamentará en los registros de **Egresos hospitalarios mediante Grupos Relacionados por el Diagnóstico (GRD)**. Esta información, proveniente del Ministerio de Salud de Chile y FONASA, se extraerá del repositorio de datos abiertos correspondiente al sector de financiamiento de la institución, utilizando la **Base de Datos GRD para el período histórico comprendido entre los años 2019 y 2024**.

El uso de un marco temporal de cinco años permitirá observar patrones consolidados y mitigar anomalías estadísticas de años particulares. Además, dado el volumen y la complejidad de la información, la base principal se integrará de manera relacional con los siguientes catálogos estandarizados:

- **Clasificación Internacional de Enfermedades (CIE-10 y CIE-9):** La implementación de estas nomenclaturas es esencial para el procesamiento de los datos. El CIE-10 permitirá normalizar los diagnósticos principales y secundarios con los que ingresa el paciente, mientras que el CIE-9 estandarizará el registro de los procedimientos médicos e intervenciones quirúrgicas aplicadas. Esta homologación es el paso crítico para asegurar que se están comparando perfiles clínicos verdaderamente equivalentes.

- **Tabla Maestra Base GRD:** Este instrumento actuará como el ponderador analítico del estudio. Al proveer los pesos relativos y los niveles de severidad precalculados para cada grupo de diagnósticos, permitirá aislar matemáticamente la complejidad inherente del cuadro médico.

Al cruzar estas fuentes, será posible estructurar un modelo de datos robusto. Esto permitirá controlar estadísticamente la variable de severidad clínica y evaluar si la varianza en los días de estada o en la cantidad de intervenciones responde a la condición del paciente o, de manera anómala, a los criterios operativos del hospital.

## 1.4 Variables de Análisis (Base de Datos GRD)

Para responder a las preguntas de investigación y aislar la complejidad médica de cada paciente, el modelo analítico utilizará las siguientes variables extraídas de los registros de egresos:

### Variable Independiente (Eje de Comparación)
- **Código de Establecimiento / Hospital:** Identificador clave que permite agrupar los registros para evaluar la varianza en los tratamientos y tiempos de internación entre los distintos recintos.

### Variables de Control (Ajuste por Severidad Clínica)
- **Códigos de Diagnóstico (CIE-10):** Diagnóstico principal y comorbilidades (secundarios). Son fundamentales para filtrar la base de datos y asegurar la comparación exclusiva entre perfiles clínicos equivalentes.
- **Código y Peso Relativo GRD:** Clasificación estandarizada y su respectivo valor numérico que representan el nivel de severidad del caso. Constituye la métrica principal para aislar matemáticamente la gravedad de la enfermedad y realizar una comparación justa entre hospitales.
- **Variables Demográficas (Edad y Sexo):** Controles estadísticos estándar necesarios para evitar sesgos en el modelado.

### Variables Dependientes (Métricas de Resultado)
- **Días de Estada:** Métrica cuantitativa utilizada para medir la duración total de la hospitalización y detectar variabilidades inexplicadas.
- **Códigos (CIE-9) y Cantidad de Procedimientos:** Tipos de intervenciones aplicadas y el volumen total de estas por egreso. Permiten identificar si existen patrones sistemáticos de mayor o menor intervención clínica frente a cuadros de idéntica complejidad.

---
# 2. Análisis Exploratorio Inicial

En esta sección se realiza la importación y limpieza preliminar de los datos. La estrategia de carga está diseñada para **no exceder la memoria RAM disponible**, utilizando:
1. **Lectura selectiva de columnas** (`usecols`): Solo se leen las ~15 columnas relevantes de las 129 totales.
2. **Procesamiento por chunks** (`chunksize`): Los archivos se leen en bloques de 50.000 filas.
3. **Tipos categóricos**: Las columnas repetitivas se convierten a `category` para reducir huella de memoria.
4. **Conversión de encoding por streaming**: Los archivos con encoding distinto a UTF-8 se convierten sin cargar todo en memoria.

### 2.1 Importación de librerías

In [ ]:
import warnings
import codecs
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 220)
sns.set_theme(style='whitegrid', context='talk')

print('Librerías cargadas correctamente.')

### 2.2 Importación y limpieza preliminar

Se definen las funciones de carga eficiente en memoria y se procesan los 6 archivos CSV del período 2019–2024.

In [ ]:
# ════════════════════════════════════════════════════════════════
# CONFIGURACIÓN
# ════════════════════════════════════════════════════════════════
DATASET_DIR = Path('DATASET-PROBLEMA8')
UTF8_DIR    = DATASET_DIR / 'utf8'
CHUNK_SIZE  = 50_000          # filas por bloque de lectura
IO_CHUNK    = 1024 * 1024     # 1 MB para conversión de encoding

ARCHIVOS = [
    'GRD_PUBLICO_2019.csv',
    'GRD_PUBLICO_2020.csv',
    'GRD_PUBLICO_2021.csv',
    'GRD_PUBLICO_EXTERNO_2022.csv',
    'GRD_PUBLICO_2023.csv',
    'GRD_PUBLICO_2024.csv',
]

# Columnas que necesitamos (las únicas que se leerán del disco)
COLUMNAS_NECESARIAS = [
    'COD_HOSPITAL',
    'CIP_ENCRIPTADO',       # 2019-2023; en 2024 se llama ID_BENEFICIARIO
    'SEXO',
    'FECHA_NACIMIENTO',
    'FECHA_INGRESO',
    'FECHAALTA',
    'DIAGNOSTICO1',         # diagnóstico principal (CIE-10)
    'DIAGNOSTICO2',         # comorbilidad 1
    'DIAGNOSTICO3',         # comorbilidad 2
    'IR_29301_COD_GRD',     # código GRD
    'IR_29301_PESO',        # peso relativo GRD
    'IR_29301_SEVERIDAD',   # nivel de severidad
    'IR_29301_MORTALIDAD',  # riesgo de mortalidad
]

# Columnas de procedimientos (CIE-9) — se cuentan, no se cargan todas como string
COLS_PROCEDIMIENTOS = [f'PROCEDIMIENTO{i}' for i in range(1, 31)]

# Encodings candidatos en orden de probabilidad
ENCODINGS = ['utf-8-sig', 'utf-8', 'utf-16', 'utf-16-le', 'utf-16-be', 'cp1252', 'latin-1']

print(f'Directorio de datos: {DATASET_DIR.resolve()}')
print(f'Archivos a procesar: {len(ARCHIVOS)}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# FUNCIONES DE CARGA EFICIENTE
# ════════════════════════════════════════════════════════════════

def convertir_a_utf8(src: Path, dst: Path) -> str:
    """Convierte un CSV a UTF-8 usando streaming (sin cargar todo en RAM).
    Retorna el encoding detectado."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return 'ya convertido'
    tmp = dst.with_suffix('.tmp')
    for enc in ENCODINGS:
        try:
            decoder = codecs.getincrementaldecoder(enc)(errors='strict')
            with src.open('rb') as fin, tmp.open('w', encoding='utf-8', newline='') as fout:
                while True:
                    raw = fin.read(IO_CHUNK)
                    if not raw:
                        break
                    fout.write(decoder.decode(raw))
                fout.write(decoder.decode(b'', final=True))
            tmp.replace(dst)
            return enc
        except UnicodeDecodeError:
            if tmp.exists():
                tmp.unlink()
    # Fallback
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    with src.open('rb') as fin, tmp.open('w', encoding='utf-8', newline='') as fout:
        while True:
            raw = fin.read(IO_CHUNK)
            if not raw:
                break
            fout.write(decoder.decode(raw))
        fout.write(decoder.decode(b'', final=True))
    tmp.replace(dst)
    return 'utf-8 (replace)'


def detectar_columnas_disponibles(path_utf8: Path) -> list:
    """Lee solo la primera fila para obtener los nombres de columna."""
    header = pd.read_csv(path_utf8, sep='|', nrows=0, encoding='utf-8')
    return list(header.columns)


def cargar_archivo_eficiente(nombre: str) -> pd.DataFrame:
    """Carga un solo archivo CSV de forma eficiente en memoria.
    Solo lee las columnas necesarias, por chunks, y calcula dias_estada."""
    src = DATASET_DIR / nombre
    dst = UTF8_DIR / nombre

    if not src.exists():
        print(f'  ❌ No encontrado: {src}')
        return pd.DataFrame()

    # 1. Convertir encoding
    enc = convertir_a_utf8(src, dst)
    print(f'  📄 {nombre} | encoding: {enc}')

    # 2. Detectar qué columnas existen en este archivo
    cols_disponibles = detectar_columnas_disponibles(dst)

    # Normalizar: 2024 usa ID_BENEFICIARIO en vez de CIP_ENCRIPTADO
    renombrar = {}
    cols_pedir = []
    for col in COLUMNAS_NECESARIAS:
        if col in cols_disponibles:
            cols_pedir.append(col)
        elif col == 'CIP_ENCRIPTADO' and 'ID_BENEFICIARIO' in cols_disponibles:
            cols_pedir.append('ID_BENEFICIARIO')
            renombrar['ID_BENEFICIARIO'] = 'CIP_ENCRIPTADO'

    # Agregar columnas de procedimientos que existan
    cols_proc_exist = [c for c in COLS_PROCEDIMIENTOS if c in cols_disponibles]
    cols_pedir += cols_proc_exist

    # 3. Leer por chunks
    chunks = []
    reader = pd.read_csv(
        dst, sep='|', usecols=cols_pedir, dtype=str,
        chunksize=CHUNK_SIZE, encoding='utf-8',
        engine='python', on_bad_lines='skip', low_memory=True
    )
    for chunk in reader:
        # Renombrar si es necesario
        if renombrar:
            chunk = chunk.rename(columns=renombrar)

        # Contar procedimientos no nulos por fila
        if cols_proc_exist:
            chunk['CANTIDAD_PROCEDIMIENTOS'] = chunk[cols_proc_exist].notna().sum(axis=1).astype('int16')
            chunk = chunk.drop(columns=cols_proc_exist)
        else:
            chunk['CANTIDAD_PROCEDIMIENTOS'] = 0

        chunks.append(chunk)

    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()

    # 4. Derivar dias_estada
    if 'FECHA_INGRESO' in df.columns and 'FECHAALTA' in df.columns:
        fi = pd.to_datetime(df['FECHA_INGRESO'], errors='coerce')
        fa = pd.to_datetime(df['FECHAALTA'], errors='coerce')
        df['DIAS_ESTADA'] = (fa - fi).dt.days

    # 5. Extraer año del egreso
    if 'FECHAALTA' in df.columns:
        df['ANIO_EGRESO'] = pd.to_datetime(df['FECHAALTA'], errors='coerce').dt.year

    # 6. Optimizar memoria: convertir a category
    for col in ['COD_HOSPITAL', 'SEXO', 'IR_29301_COD_GRD', 'IR_29301_SEVERIDAD', 'IR_29301_MORTALIDAD']:
        if col in df.columns:
            df[col] = df[col].astype('category')

    # Convertir peso GRD a numérico (usa coma decimal en Chile)
    if 'IR_29301_PESO' in df.columns:
        df['IR_29301_PESO'] = (
            df['IR_29301_PESO'].str.replace(',', '.', regex=False)
            .pipe(pd.to_numeric, errors='coerce')
        )

    df['ARCHIVO_ORIGEN'] = nombre
    print(f'     ✅ {len(df):>10,} filas | {df.memory_usage(deep=True).sum() / 1e6:.1f} MB en RAM')
    return df

print('Funciones de carga definidas.')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CARGA DE TODOS LOS ARCHIVOS
# ════════════════════════════════════════════════════════════════
print('Iniciando carga eficiente de datos...\n')

lista_dfs = []
for nombre in ARCHIVOS:
    df_tmp = cargar_archivo_eficiente(nombre)
    if not df_tmp.empty:
        lista_dfs.append(df_tmp)
    del df_tmp
    gc.collect()

if not lista_dfs:
    raise ValueError('No se pudo cargar ningún archivo. Revisa las rutas.')

df = pd.concat(lista_dfs, ignore_index=True)
del lista_dfs
gc.collect()

print(f'\n{"="*60}')
print(f'DATASET CONSOLIDADO')
print(f'  Registros totales : {len(df):>12,}')
print(f'  Columnas          : {len(df.columns):>12}')
print(f'  Memoria en RAM    : {df.memory_usage(deep=True).sum() / 1e6:>12.1f} MB')
print(f'{"="*60}')
display(df.head())

### 2.3 Análisis descriptivo de variables clave

In [ ]:
# Limpieza: eliminar registros sin datos críticos y outliers
n_original = len(df)

# Asegurar tipos numéricos
df['DIAS_ESTADA'] = pd.to_numeric(df['DIAS_ESTADA'], errors='coerce')

# Eliminar registros sin hospital, sin GRD, sin días de estada
df = df.dropna(subset=['COD_HOSPITAL', 'IR_29301_COD_GRD', 'DIAS_ESTADA']).copy()
df = df[df['DIAS_ESTADA'] >= 0].copy()

# Eliminar outliers extremos (percentil 99)
p99 = df['DIAS_ESTADA'].quantile(0.99)
df = df[df['DIAS_ESTADA'] <= p99].copy()

n_limpio = len(df)
print(f'Registros originales : {n_original:>10,}')
print(f'Registros después    : {n_limpio:>10,}')
print(f'Eliminados           : {n_original - n_limpio:>10,}')
print(f'Corte outlier p99    : {p99:.0f} días')

In [ ]:
# Estadísticas descriptivas globales de las variables clave
print('=== Distribución de Días de Estada ===')
display(df['DIAS_ESTADA'].describe().to_frame().T)

print('\n=== Distribución de Sexo ===')
display(df['SEXO'].value_counts().to_frame())

print('\n=== Distribución de Severidad GRD ===')
display(df['IR_29301_SEVERIDAD'].value_counts().sort_index().to_frame())

print('\n=== Distribución de Mortalidad GRD ===')
display(df['IR_29301_MORTALIDAD'].value_counts().sort_index().to_frame())

print('\n=== Top 10 GRD más frecuentes ===')
display(df['IR_29301_COD_GRD'].value_counts().head(10).to_frame())

### 2.4 Identificación de patrones iniciales

In [ ]:
# Visualización: distribución de días de estada
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(df['DIAS_ESTADA'].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribución de Días de Estada')
axes[0].set_xlabel('Días')
axes[0].set_ylabel('Frecuencia')

sev_counts = df['IR_29301_SEVERIDAD'].value_counts().sort_index()
axes[1].bar(sev_counts.index.astype(str), sev_counts.values, color='coral', edgecolor='white')
axes[1].set_title('Distribución de Severidad GRD')
axes[1].set_xlabel('Nivel de Severidad')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

### 2.5 Documentación de hallazgos preliminares

In [ ]:
# Resumen de completitud por columna
completitud = (df.notna().sum() / len(df) * 100).round(1)
resumen_comp = pd.DataFrame({
    'Columna': completitud.index,
    'Completitud (%)': completitud.values,
    'Filas con dato': df.notna().sum().values
}).sort_values('Completitud (%)', ascending=False).reset_index(drop=True)

print('Completitud por columna:')
display(resumen_comp)

print(f'\nMemoria actual del DataFrame: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

---
# 3. Profundización del Contexto

### 3.1 Investigación sobre el Dataset

Los datos provienen de los **Egresos Hospitalarios clasificados por Grupos Relacionados por el Diagnóstico (GRD)**, publicados por FONASA y el Ministerio de Salud de Chile. Cada registro representa un egreso hospitalario individual del sistema público.

**Características técnicas del dataset:**
- **Período**: 2019–2024 (6 archivos anuales)
- **Formato**: CSV delimitado por pipe (`|`)
- **Encodings mixtos**: UTF-8-sig (2019–2021), UTF-16 (2022–2023), Latin-1 (2024)
- **Columnas originales**: 129 por archivo
- **Columnas relevantes cargadas**: ~15 (optimización de memoria)

### 3.2 Glosario de términos

| Término | Definición |
|---------|------------|
| **GRD** | Grupos Relacionados por el Diagnóstico. Sistema de clasificación que agrupa egresos hospitalarios en categorías clínicamente coherentes y de consumo de recursos similar. |
| **CIE-10** | Clasificación Internacional de Enfermedades, 10ª revisión. Nomenclatura estándar para diagnósticos médicos. |
| **CIE-9** | Clasificación Internacional de Enfermedades, 9ª revisión. Utilizada aquí para codificar procedimientos médicos e intervenciones quirúrgicas. |
| **Peso Relativo GRD** | Valor numérico que representa la complejidad relativa y el consumo esperado de recursos de un grupo diagnóstico específico. |
| **Severidad** | Nivel de gravedad clínica del caso (1–4), precalculado en el sistema GRD. |
| **Mortalidad** | Riesgo de mortalidad asociado al caso (1–4), precalculado en el sistema GRD. |
| **Días de Estada** | Duración total de la hospitalización, calculada como la diferencia entre la fecha de alta y la fecha de ingreso. |
| **FONASA** | Fondo Nacional de Salud. Organismo público de financiamiento de salud en Chile. |

### 3.3 Planteamiento de preguntas de investigación preliminares

1. ¿Presentan los hospitales diferencias significativas en los días de estada promedio para un mismo GRD?
2. ¿Varía la cantidad de procedimientos realizados entre hospitales para pacientes con severidad clínica equivalente?
3. ¿Existen hospitales con patrones sistemáticamente distintos al resto de la red?
4. ¿Las diferencias encontradas persisten después de controlar por sexo, edad y peso relativo GRD?

---
# 4. Definición del Proyecto

### 4.1 Selección de pregunta de investigación final

> **¿Existe una variabilidad significativa en los protocolos de tratamiento y tiempos de estada para pacientes con diagnósticos idénticos, dependiendo de si son atendidos en uno u otro recinto de la red hospitalaria pública?**

### 4.2 Justificación de relevancia y factibilidad

- **Relevancia clínica**: Identificar hospitales con prácticas divergentes permite focalizar intervenciones de mejora.
- **Relevancia de gestión**: Las diferencias en días de estada impactan directamente la disponibilidad de camas y costos operativos.
- **Factibilidad**: Se dispone de >5 millones de registros con variables de severidad precalculadas que permiten realizar comparaciones justas.

### 4.3 Definición de variables a analizar

| Rol | Variable | Columna en datos |
|-----|----------|------------------|
| **Independiente** | Hospital | `COD_HOSPITAL` |
| **Dependiente** | Días de estada | `DIAS_ESTADA` (derivada) |
| **Dependiente** | Cantidad de procedimientos | `CANTIDAD_PROCEDIMIENTOS` (derivada) |
| **Control** | Diagnóstico principal (CIE-10) | `DIAGNOSTICO1` |
| **Control** | Código GRD | `IR_29301_COD_GRD` |
| **Control** | Peso relativo GRD | `IR_29301_PESO` |
| **Control** | Severidad | `IR_29301_SEVERIDAD` |
| **Control** | Sexo | `SEXO` |

### 4.4 Plan de análisis detallado

1. Seleccionar un GRD de alta prevalencia para asegurar volumen estadístico.
2. Calcular estimadores puntuales (media, mediana, varianza) de días de estada y procedimientos por hospital.
3. Construir intervalos de confianza al 95% para la media de días de estada por hospital.
4. Evaluar normalidad de la distribución (Shapiro-Wilk sobre submuestra).
5. Aplicar prueba no paramétrica de Kruskal-Wallis para comparación entre hospitales.
6. Interpretar resultados y documentar conclusiones.

---
# 5. Estimación Estadística

Selección de un GRD focal, cálculo de estimadores puntuales y comparación entre grupos hospitalarios.

### 5.1 Selección de parámetros relevantes para estimar

In [ ]:
# Selección del GRD focal: el más frecuente en la base
grd_counts = df['IR_29301_COD_GRD'].value_counts()
SELECTED_GRD = grd_counts.index[0]

df_focus = df[df['IR_29301_COD_GRD'] == SELECTED_GRD].copy()

print(f'GRD seleccionado     : {SELECTED_GRD}')
print(f'Registros            : {len(df_focus):,}')
print(f'Hospitales únicos    : {df_focus["COD_HOSPITAL"].nunique()}')
print(f'\nTop 5 GRD por frecuencia:')
display(grd_counts.head().to_frame())

### 5.2 Cálculo de estimadores puntuales

In [ ]:
# Estadística descriptiva por hospital para el GRD seleccionado
summary = (
    df_focus.groupby('COD_HOSPITAL', observed=True)
    .agg(
        n_casos        = ('DIAS_ESTADA', 'count'),
        dias_media     = ('DIAS_ESTADA', 'mean'),
        dias_mediana   = ('DIAS_ESTADA', 'median'),
        dias_std       = ('DIAS_ESTADA', 'std'),
        dias_varianza  = ('DIAS_ESTADA', 'var'),
        proc_media     = ('CANTIDAD_PROCEDIMIENTOS', 'mean'),
        proc_mediana   = ('CANTIDAD_PROCEDIMIENTOS', 'median'),
        peso_grd_media = ('IR_29301_PESO', 'mean'),
    )
    .reset_index()
    .sort_values('n_casos', ascending=False)
)

print(f'Estimadores puntuales por hospital (GRD = {SELECTED_GRD}):')
display(summary.head(20))

### 5.3 Comparación entre grupos de hospitales

In [ ]:
# Visualización: boxplot de días de estada por hospital (top 15)
MIN_CASOS = 30
hospitales_validos = summary[summary['n_casos'] >= MIN_CASOS]['COD_HOSPITAL']
df_plot = df_focus[df_focus['COD_HOSPITAL'].isin(hospitales_validos)].copy()

top15 = df_plot['COD_HOSPITAL'].value_counts().head(15).index
df_plot_top = df_plot[df_plot['COD_HOSPITAL'].isin(top15)].copy()

fig, axes = plt.subplots(2, 1, figsize=(18, 14))

# Boxplot días de estada
sns.boxplot(
    data=df_plot_top, x='COD_HOSPITAL', y='DIAS_ESTADA',
    showfliers=False, palette='viridis', ax=axes[0]
)
axes[0].set_title(f'Distribución de Días de Estada por Hospital (GRD = {SELECTED_GRD})')
axes[0].set_xlabel('Hospital')
axes[0].set_ylabel('Días de Estada')
axes[0].tick_params(axis='x', rotation=45)

# Barplot promedio de procedimientos
avg_proc = (
    df_plot_top.groupby('COD_HOSPITAL', observed=True, as_index=False)['CANTIDAD_PROCEDIMIENTOS']
    .mean()
    .sort_values('CANTIDAD_PROCEDIMIENTOS', ascending=False)
)
sns.barplot(
    data=avg_proc, x='COD_HOSPITAL', y='CANTIDAD_PROCEDIMIENTOS',
    palette='magma', ax=axes[1]
)
axes[1].set_title(f'Promedio de Procedimientos por Hospital (GRD = {SELECTED_GRD})')
axes[1].set_xlabel('Hospital')
axes[1].set_ylabel('Promedio de Procedimientos')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 5.4 Documentación de resultados iniciales

In [ ]:
# Resumen estadístico global del GRD focal
print(f'=== Resumen del GRD {SELECTED_GRD} ===')
print(f'  Hospitales con >= {MIN_CASOS} casos: {len(hospitales_validos)}')
print(f'  Días de estada — Media global   : {df_focus["DIAS_ESTADA"].mean():.2f}')
print(f'  Días de estada — Mediana global  : {df_focus["DIAS_ESTADA"].median():.1f}')
print(f'  Días de estada — Desv. estándar  : {df_focus["DIAS_ESTADA"].std():.2f}')
print(f'  Procedimientos — Media global    : {df_focus["CANTIDAD_PROCEDIMIENTOS"].mean():.2f}')
print(f'\n  Rango de medias por hospital:')
print(f'    Mínimo: {summary["dias_media"].min():.2f} días')
print(f'    Máximo: {summary["dias_media"].max():.2f} días')
print(f'    Diferencia: {summary["dias_media"].max() - summary["dias_media"].min():.2f} días')

---
# 6. Análisis Inferencial Inicial

Construcción de intervalos de confianza, pruebas de hipótesis y visualización de resultados.

### 6.1 Construcción de intervalos de confianza para variables clave

In [ ]:
# Intervalos de confianza al 95% para la media de días de estada por hospital
from scipy.stats import t as t_dist

alpha = 0.05
ic_list = []

for hosp, grupo in df_plot[df_plot['COD_HOSPITAL'].isin(top15)].groupby('COD_HOSPITAL', observed=True):
    x = grupo['DIAS_ESTADA'].dropna()
    n = len(x)
    if n < 2:
        continue
    media = x.mean()
    se = x.std(ddof=1) / np.sqrt(n)
    t_crit = t_dist.ppf(1 - alpha/2, df=n-1)
    ic_list.append({
        'Hospital': hosp,
        'n': n,
        'Media': round(media, 2),
        'IC_inferior': round(media - t_crit * se, 2),
        'IC_superior': round(media + t_crit * se, 2),
    })

df_ic = pd.DataFrame(ic_list).sort_values('Media', ascending=False)
print(f'Intervalos de confianza al {(1-alpha)*100:.0f}% — Días de Estada (GRD = {SELECTED_GRD}):')
display(df_ic)

### 6.2 Visualización de intervalos

In [ ]:
# Gráfico de intervalos de confianza
fig, ax = plt.subplots(figsize=(14, 8))

df_ic_sorted = df_ic.sort_values('Media')
y_pos = range(len(df_ic_sorted))

ax.barh(
    y_pos, df_ic_sorted['Media'],
    xerr=[df_ic_sorted['Media'] - df_ic_sorted['IC_inferior'],
          df_ic_sorted['IC_superior'] - df_ic_sorted['Media']],
    color='steelblue', alpha=0.7, edgecolor='white',
    error_kw={'capsize': 4, 'elinewidth': 1.5, 'capthick': 1.5}
)
ax.set_yticks(y_pos)
ax.set_yticklabels(df_ic_sorted['Hospital'].astype(str))
ax.set_xlabel('Días de Estada (media ± IC 95%)')
ax.set_title(f'Intervalos de Confianza por Hospital (GRD = {SELECTED_GRD})')
ax.axvline(df_focus['DIAS_ESTADA'].mean(), color='red', linestyle='--', label='Media global')
ax.legend()
plt.tight_layout()
plt.show()

### 6.3 Interpretación de resultados — Pruebas de hipótesis

In [ ]:
# Test de normalidad (Shapiro-Wilk sobre submuestra)
x_dias = df_focus['DIAS_ESTADA'].dropna().values

if len(x_dias) > 5000:
    rng = np.random.default_rng(42)
    x_norm = rng.choice(x_dias, size=5000, replace=False)
    nota = '(submuestra aleatoria n=5.000)'
else:
    x_norm = x_dias
    nota = '(muestra completa)'

w_stat, p_norm = stats.shapiro(x_norm)

print(f'=== Test de Normalidad — Shapiro-Wilk {nota} ===')
print(f'  Estadístico W : {w_stat:.6f}')
print(f'  P-value       : {p_norm:.6e}')
if p_norm < alpha:
    print(f'  → Se RECHAZA H0 de normalidad (α={alpha}). Distribución no normal.')
    print(f'  → Se justifica el uso de pruebas no paramétricas.')
else:
    print(f'  → No se rechaza H0 de normalidad.')

In [ ]:
# Test de Kruskal-Wallis entre hospitales
groups = [
    g['DIAS_ESTADA'].dropna().values
    for _, g in df_plot.groupby('COD_HOSPITAL', observed=True)
    if len(g) >= MIN_CASOS
]

if len(groups) < 2:
    print('⚠️ No hay suficientes hospitales con el mínimo de casos para Kruskal-Wallis.')
else:
    h_stat, p_kw = stats.kruskal(*groups)
    print(f'=== Test de Kruskal-Wallis ===')
    print(f'  Hospitales comparados : {len(groups)}')
    print(f'  Estadístico H         : {h_stat:.4f}')
    print(f'  P-value               : {p_kw:.6e}')
    print()
    if p_kw < alpha:
        print(f'  → Se RECHAZA H0 (α={alpha}).')
        print(f'  → Existen diferencias SIGNIFICATIVAS en días de estada entre hospitales,')
        print(f'    incluso controlando por diagnóstico (mismo GRD = {SELECTED_GRD}).')
    else:
        print(f'  → No se rechaza H0. No hay evidencia suficiente de diferencias.')

### 6.4 Preparación para avance del proyecto

**Hallazgos hasta este punto:**

- Se logró cargar y consolidar exitosamente los ~4 GB de datos sin agotar la memoria RAM, mediante lectura selectiva de columnas, procesamiento por chunks y tipos categóricos.
- Se derivó la variable `DIAS_ESTADA` a partir de las fechas de ingreso y alta.
- Se identificaron los GRD de mayor prevalencia y se calcularon estimadores puntuales por hospital.
- Los intervalos de confianza al 95% muestran diferencias visibles entre hospitales.
- Las pruebas de hipótesis (Shapiro-Wilk y Kruskal-Wallis) proveen evidencia estadística sobre la variabilidad.

**Próximos pasos sugeridos:**

1. Extender el análisis a múltiples GRDs para generalizar las conclusiones.
2. Incorporar modelos multivariados que controlen simultáneamente por severidad, sexo y edad.
3. Análisis post-hoc (Dunn) para identificar qué pares de hospitales difieren significativamente.
4. Evaluación longitudinal (tendencia año a año).
5. Redacción final de conclusiones y limitaciones.